<a href="https://colab.research.google.com/github/arthurkennedy/Team-14/blob/Burn-Care-Capacity-Analysis/Competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Determine if nearby burn centers have enough capacity.

# Imports

In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Data Cleaning

In [2]:
file_path = Path("NIRD_20230130_Database_Hackathon.xlsx")
sheet_name = "Data Table NIRD 20230130"
data= pd.read_excel(file_path, sheet_name=sheet_name)
data.head()

,STATE_FULL,STATE,COUNTY,ADDRESS,CITY,ZIP_CODE,AHA_ID,HOSPITAL_NAME,TOTAL_BEDS,BURN_BEDS,...,BURN_PEDS,ACS_VERIFIED,ADULT_TRAUMA_L1,ADULT_TRAUMA_L2,PEDS_TRAUMA_L1,PEDS_TRAUMA_L2,ABA_VERIFIED,TC_STATE_DESIGNATED,BC_STATE_DESIGNATED,PHONE
0,Alaska,AK,Anchorage,4315 Diplomacy Dr,Anchorage,99508,6940010.0,Alaska Native Medical Center,173,NaN,...,NaN,Yes,NaN,1.0,NaN,1.0,NaN,Yes,NaN,(907) 563-2662
1,Alaska,AK,Anchorage,3200 Providence Dr,Anchorage,99508,6940020.0,Providence Alaska Medical Center/Children's Ho...,401,NaN,...,NaN,Yes,NaN,1.0,NaN,1.0,NaN,Yes,NaN,(907) 562-2211
2,Alabama,AL,Houston,1108 Ross Clark Cir,Dothan,36301,6530373.0,Southeast Alabama Medical Center,387,NaN,...,NaN,No,NaN,1.0,NaN,NaN,NaN,Yes,NaN,(334) 793-8111
3,Alabama,AL,Jefferson,619 19th St S,Birmingham,35233,6530304.0,University of Alabama at Birmingham Hospital (...,1157,28.0,...,NaN,Yes,1.0,NaN,NaN,NaN,No,Yes,No,(205) 934-3411
4,Alabama,AL,Jefferson,1600 7th Ave South,Birmingham,35233,6530170.0,Children's of Alabama (Children's of Alabama B...,351,6.0,...,1.0,No,NaN,NaN,1.0,NaN,No,Yes,No,(205) 638-9100


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 635 entries, 0 to 634
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   STATE_FULL           635 non-null    object 
 1   STATE                635 non-null    object 
 2   COUNTY               635 non-null    object 
 3   ADDRESS              635 non-null    object 
 4   CITY                 635 non-null    object 
 5   ZIP_CODE             635 non-null    int64  
 6   AHA_ID               634 non-null    float64
 7   HOSPITAL_NAME        635 non-null    object 
 8   TOTAL_BEDS           635 non-null    int64  
 9   BURN_BEDS            130 non-null    float64
 10  TRAUMA_ADULT         565 non-null    float64
 11  TRAUMA_PEDS          146 non-null    float64
 12  BURN_ADULT           120 non-null    float64
 13  BURN_PEDS            89 non-null     float64
 14  ACS_VERIFIED         617 non-null    object 
 15  ADULT_TRAUMA_L1      229 non-null    flo

Compute missing data

In [4]:
# 1. Fill numerical columns with 0 (so you can do math)
num_cols = ['BURN_BEDS', 'TOTAL_BEDS', 'BURN_ADULT', 'BURN_PEDS',
            'TRAUMA_ADULT', 'TRAUMA_PEDS', 'ADULT_TRAUMA_L1',
            'ADULT_TRAUMA_L2', 'PEDS_TRAUMA_L1', 'PEDS_TRAUMA_L2']
data[num_cols] = data[num_cols].fillna(0)

# 2. Fill categorical/text columns with "No" or "Unknown"
cat_cols = ['ABA_VERIFIED', 'ACS_VERIFIED', 'BC_STATE_DESIGNATED', 'TC_STATE_DESIGNATED']
data[cat_cols] = data[cat_cols].fillna("No")

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 635 entries, 0 to 634
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   STATE_FULL           635 non-null    object 
 1   STATE                635 non-null    object 
 2   COUNTY               635 non-null    object 
 3   ADDRESS              635 non-null    object 
 4   CITY                 635 non-null    object 
 5   ZIP_CODE             635 non-null    int64  
 6   AHA_ID               634 non-null    float64
 7   HOSPITAL_NAME        635 non-null    object 
 8   TOTAL_BEDS           635 non-null    int64  
 9   BURN_BEDS            635 non-null    float64
 10  TRAUMA_ADULT         635 non-null    float64
 11  TRAUMA_PEDS          635 non-null    float64
 12  BURN_ADULT           635 non-null    float64
 13  BURN_PEDS            635 non-null    float64
 14  ACS_VERIFIED         635 non-null    object 
 15  ADULT_TRAUMA_L1      635 non-null    flo

In [6]:
len(data["STATE"].unique())

50

In [7]:
print(data['STATE_FULL'].unique())

['Alaska' 'Alabama' 'Arizona' 'Arkansas' 'California' 'Colorado'
 'Connecticut' 'Delaware' 'District of Columbia' 'Florida' 'Georgia'
 'Hawaii' 'Idaho' 'Illinois' 'Indiana' 'Iowa' 'Kansas' 'Kentucky'
 'Louisiana' 'Maine' 'Maryland' 'Massachusetts' 'Michigan' 'Minnesota '
 'Mississippi' 'Missouri' 'Montana' 'Nebraska' 'Nevada' 'New Hampshire'
 'New Jersey' 'New Mexico' 'New York' 'North Carolina' 'North Dakota'
 'Ohio' 'Oklahoma' 'Oregon' 'Pennsylvania' 'Rhode Island' 'South Carolina'
 'South Dakota' 'Tennessee' 'Texas' 'Utah' 'Vermont' 'Virginia'
 'Washington' 'West Virginia' 'Wisconsin']


In [8]:
len(data['STATE_FULL'].unique())

50

In [9]:
# Calculate medians while ignoring the 'NaN' values
median_total = data['BURN_BEDS'].median()
median_adult = data['BURN_ADULT'].median()
median_peds = data['BURN_PEDS'].median()

print(f"Median Total Burn Beds: {median_total}")
print(f"Median Adult Burn Beds: {median_adult}")
print(f"Median Pediatric Burn Beds: {median_peds}")

Median Total Burn Beds: 0.0
Median Adult Burn Beds: 0.0
Median Pediatric Burn Beds: 0.0


# INFO

1. The "Median" Benchmark
According to the NIRD (2023) paper, the median number of burn ICU beds in a U.S. burn center is 11.

Any state with fewer than 10 beds is operating below the national average for a single center.

If an entire state has fewer than 10 beds, it means that state likely has only one very small unit or, in many cases, none at all.

2. The "Disaster Threshold" **(The 5-Patient Rule)**
The NIRD research notes a very specific safety limit: Most burn centers are restricted to handling only 5 large burn admissions at one time during a mass casualty event or disaster.

Why 10? If a state has 10 beds, and they are already 50% full with daily patients (which is common), a single house fire or industrial accident involving just 5 new patients would completely fill the state's capacity.

States with < 10 beds are considered "Critically Low" because they have zero "surge capacity." They cannot handle a local disaster without immediately needing to fly patients to other states.

3. Identifying "Burn Deserts"
By filtering for < 10, the code captures two types of high-risk regions:

**Total Deserts** (0 beds): The 8 states that have no specialized care at all (like Alaska or Mississippi).

Fragile States (1–9 beds): States like West Virginia (4 beds) or Maine (6 beds). These states technically have a burn center, but it is so small that it lacks the resources to be "ABA-Verified" or to handle more than a couple of patients at once.

# Task 1: Burn Beds per Region (State)

Hospitals had a median of 492 staffed beds; 79,449 ER visits; and 18,437 discharges. Burn centers had a median of 11 burn ICU beds and 3285 hospital days per year

In [10]:
# Sum burn beds by state
state_capacity = data.groupby('STATE_FULL')['BURN_BEDS'].sum().reset_index()
state_capacity = state_capacity.sort_values(by='BURN_BEDS', ascending=False)

# Identify regions with LOW capacity (less than 10 beds or 0)
low_capacity_regions = state_capacity[state_capacity['BURN_BEDS'] <= 10]
print(f"there are {len(low_capacity_regions)} states that low capacity")
print("States with critically low burn bed capacity:")
print(low_capacity_regions)

there are 15 states that low capacity
States with critically low burn bed capacity:
       STATE_FULL  BURN_BEDS
3        Arkansas       10.0
31     New Mexico       10.0
6     Connecticut        9.0
45        Vermont        9.0
11         Hawaii        7.0
19          Maine        6.0
48  West Virginia        4.0
7        Delaware        0.0
1          Alaska        0.0
24    Mississippi        0.0
29  New Hampshire        0.0
26        Montana        0.0
41   South Dakota        0.0
34   North Dakota        0.0
39   Rhode Island        0.0


There are 8 states that do not have any burn beds at all

In [11]:
# Identify regions with HIGH capacity (less than 10 beds or 0)
high_capacity_regions = state_capacity[state_capacity['BURN_BEDS'] > 10]

print(f"there are {len(high_capacity_regions)} states that high capacity")
print("States with critically high burn bed capacity:")
print(high_capacity_regions)



there are 35 states that high capacity
States with critically high burn bed capacity:
              STATE_FULL  BURN_BEDS
43                 Texas      206.0
32              New York      143.0
4             California      140.0
10               Georgia      129.0
35                  Ohio       94.0
25              Missouri       93.0
9                Florida       85.0
38          Pennsylvania       83.0
2                Arizona       80.0
13              Illinois       77.0
22              Michigan       69.0
46              Virginia       68.0
33        North Carolina       60.0
0                Alabama       52.0
5               Colorado       52.0
21         Massachusetts       52.0
42             Tennessee       51.0
23            Minnesota        50.0
18             Louisiana       47.0
8   District of Columbia       41.0
14               Indiana       41.0
47            Washington       40.0
36              Oklahoma       33.0
28                Nevada       31.0
16            

There are about 4 states that have more than 100 burn beds when the Median beds in the country are only 11

In [12]:
import plotly.express as px

# 1. Clean the data (Crucial: Strip spaces FIRST so mapping works)
data['STATE_FULL'] = data['STATE_FULL'].str.strip()
data['BURN_BEDS'] = data['BURN_BEDS'].fillna(0)

# 2. Group by state
state_map_data = data.groupby('STATE_FULL')['BURN_BEDS'].sum().reset_index()

# 3. Add mapping for 2-letter codes
us_state_to_abbrev = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA', 'Colorado': 'CO',
    'Connecticut': 'CT', 'Delaware': 'DE', 'District of Columbia': 'DC', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD', 'Massachusetts': 'MA',
    'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT',
    'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM',
    'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD',
    'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY'
}
state_map_data['state_code'] = state_map_data['STATE_FULL'].map(us_state_to_abbrev)

# 4. Create the Figure
fig = px.choropleth(
    state_map_data,
    locations='state_code',
    locationmode='USA-states',
    color='BURN_BEDS',
    scope='usa',
    color_continuous_scale="Reds",
    # FIX: Use range_color to start at 0 instead of using a midpoint
    range_color=[0, state_map_data['BURN_BEDS'].max()],
    labels={'BURN_BEDS': 'Total Burn Beds'},
    title='National Burn Bed Distribution'
)

fig.show()

Wymong is not in the list as states.

This heatmap shows a huge gap in care across the country, splitting the U.S. into "Hubs" and "Deserts." A few "Hub" states like Texas (206 beds) and New York (143 beds) have a massive amount of resources, while 15 states have almost nothing (10 beds or fewer). Most alarming are the 8 states like Mississippi and Montana that have zero beds, meaning patients there have no local choice but to be flown to another state for specialized treatment. Essentially, the map proves that where you live determines if you have a hospital nearby or if you have to travel hundreds of miles for help.

#**Task 2: Burn Beds per Trauma Hospital**

In [13]:
# Create a flag: Is this a Trauma Center? (Levels 1 or 2)
data['IS_TRAUMA'] = (data['ADULT_TRAUMA_L1'] > 0) | (data['ADULT_TRAUMA_L2'] > 0)

# Filter for trauma hospitals and see their burn capacity
trauma_hospitals = data[data['IS_TRAUMA'] == True].copy()
trauma_hospitals['BURN_CAPACITY_PERCENT'] = (trauma_hospitals['BURN_BEDS'] / trauma_hospitals['TOTAL_BEDS']) * 100

print(f"Average Burn Capacity in Trauma Hospitals: {trauma_hospitals['BURN_CAPACITY_PERCENT'].mean():.2f}%")

Average Burn Capacity in Trauma Hospitals: 0.57%


In [14]:
# Identify the type of facility
def get_type(row):
    if row['ADULT_TRAUMA_L1'] > 0: return 'Level 1 Trauma'
    if row['ADULT_TRAUMA_L2'] > 0: return 'Level 2 Trauma'
    return 'Non-Trauma'

data['TRAUMA_LEVEL'] = data.apply(get_type, axis=1)

# Grouping to see the Average Burn Beds by Trauma Level
trauma_analysis = data.groupby('TRAUMA_LEVEL').agg({
    'BURN_BEDS': ['sum', 'mean'],
    'TOTAL_BEDS': 'sum'
}).reset_index()

print(trauma_analysis)

     TRAUMA_LEVEL BURN_BEDS           TOTAL_BEDS
                        sum      mean        sum
0  Level 1 Trauma    1364.0  5.956332     144724
1  Level 2 Trauma     301.0  0.895833     122735
2      Non-Trauma     415.0  5.928571      20984


In [15]:
# 2. Categorize each hospital by its highest Trauma Level
def get_trauma_label(row):
    if row['ADULT_TRAUMA_L1'] > 0:
        return 'Level 1 Trauma'
    elif row['ADULT_TRAUMA_L2'] > 0:
        return 'Level 2 Trauma'
    else:
        return 'Non-Trauma / Other'

data['TRAUMA_LEVEL_TYPE'] = data.apply(get_trauma_label, axis=1)

# 3. Group by State and Trauma Level
state_trauma_analysis = data.groupby(['STATE_FULL', 'TRAUMA_LEVEL_TYPE'])['BURN_BEDS'].sum().unstack(fill_value=0)

# 4. Add a Total column and sort to see the biggest gaps
state_trauma_analysis['Total_State_Burn_Beds'] = state_trauma_analysis.sum(axis=1)
state_trauma_analysis = state_trauma_analysis.sort_values(by='Total_State_Burn_Beds')

# 5. Identify the "Infrastructure Gap"
# These are states that have Level 1 Trauma centers but 0 Burn Beds in them
gap_states = state_trauma_analysis[(state_trauma_analysis['Level 1 Trauma'] == 0) &
                                   (state_trauma_analysis['Total_State_Burn_Beds'] > 0)]

print("Analysis of Burn Beds by State and Trauma Level:")
print(state_trauma_analysis)

if not gap_states.empty:
    print("\nStates with Level 1 Trauma Centers but NO Burn Beds in those centers:")
    print(gap_states.index.tolist())

Analysis of Burn Beds by State and Trauma Level:
TRAUMA_LEVEL_TYPE     Level 1 Trauma  Level 2 Trauma  Non-Trauma / Other  \
STATE_FULL                                                                 
Alaska                           0.0             0.0                 0.0   
Delaware                         0.0             0.0                 0.0   
New Hampshire                    0.0             0.0                 0.0   
Montana                          0.0             0.0                 0.0   
Mississippi                      0.0             0.0                 0.0   
South Dakota                     0.0             0.0                 0.0   
Rhode Island                     0.0             0.0                 0.0   
North Dakota                     0.0             0.0                 0.0   
West Virginia                    0.0             4.0                 0.0   
Maine                            6.0             0.0                 0.0   
Hawaii                           0.0   

## Low capacity burn beds by Trauma level

In [16]:
# # 1. Group by State
# # We will list the hospitals and sum their burn beds for each state
# state_grouped_trauma = trauma_in_low_cap.groupby('STATE_FULL').agg({
#     #'HOSPITAL_NAME': lambda x: ', '.join(x),  # Combine all hospital names into one string
#     'BURN_BEDS': 'sum',                        # Total burn beds in these trauma centers
#     'ADULT_TRAUMA_L1': 'sum',                  # Count how many L1s are in this group
#     'ADULT_TRAUMA_L2': 'sum'                   # Count how many L2s are in this group
# }).reset_index()

# # 2. Rename columns for a professional look
# state_grouped_trauma.columns = [
#     'State',
#     #'Trauma Hospitals',
#     'Total Trauma Burn Beds',
#     'L1 Count',
#     'L2 Count'
# ]

# # 3. Sort by State name for easy reading
# state_grouped_trauma = state_grouped_trauma.sort_values(by='State')

# print("Grouped Trauma Centers by State (Low Capacity Regions):")
# print(state_grouped_trauma)

In [17]:
# 1. Create logic for L1, L2, and Non-Trauma Burn Beds
# L1 Beds
data['L1_Burn_Beds'] = data.apply(lambda x: x['BURN_BEDS'] if x['ADULT_TRAUMA_L1'] > 0 else 0, axis=1)

# L2 Beds
data['L2_Burn_Beds'] = data.apply(lambda x: x['BURN_BEDS'] if x['ADULT_TRAUMA_L2'] > 0 else 0, axis=1)

# Non-Trauma Beds (If it's NOT L1 and NOT L2, it's Non-Trauma)
data['Non_Trauma_Burn_Beds'] = data.apply(
    lambda x: x['BURN_BEDS'] if (x['ADULT_TRAUMA_L1'] == 0 and x['ADULT_TRAUMA_L2'] == 0) else 0, axis=1
)

# 2. Filter for your 15 low capacity states
low_cap_state_names = low_capacity_regions['STATE_FULL'].tolist()
filtered_data = data[data['STATE_FULL'].isin(low_cap_state_names)]

# 3. Group and aggregate
state_breakdown = filtered_data.groupby('STATE_FULL').agg({
    'ADULT_TRAUMA_L1': 'sum',         # Total L1 Centers
    'L1_Burn_Beds': 'sum',            # Total Beds in L1s
    'ADULT_TRAUMA_L2': 'sum',         # Total L2 Centers
    'L2_Burn_Beds': 'sum',            # Total Beds in L2s
    'Non_Trauma_Burn_Beds': 'sum'     # Total Beds in everything else
}).reset_index()

# 4. Rename columns for the final report
state_breakdown.columns = [
    'State Name',
    'L1 Count',
    'L1 Burn Beds',
    'L2 Count',
    'L2 Burn Beds',
    'Non-Trauma Burn Beds'
]

print("Low capacity states:")
print(state_breakdown.to_string(index=False))

Low capacity states:
   State Name  L1 Count  L1 Burn Beds  L2 Count  L2 Burn Beds  Non-Trauma Burn Beds
       Alaska       0.0           0.0       2.0           0.0                   0.0
     Arkansas       1.0           0.0       4.0           0.0                  10.0
  Connecticut       3.0           0.0       8.0           9.0                   0.0
     Delaware       1.0           0.0       0.0           0.0                   0.0
       Hawaii       1.0           0.0       1.0           0.0                   7.0
        Maine       1.0           6.0       1.0           0.0                   0.0
  Mississippi       1.0           0.0       3.0           0.0                   0.0
      Montana       0.0           0.0       4.0           0.0                   0.0
New Hampshire       1.0           0.0       3.0           0.0                   0.0
   New Mexico       1.0          10.0       0.0           0.0                   0.0
 North Dakota       1.0           0.0       5.0        

L1 hospitals: Only New Mexico, Maine and Vermont have burn beds


l2 hospitals: Connecticut, West Virginia


Non Trauma hospitals: Arkansas and Hawaii

This data highlights a structural failure in the "Desert States," where the highest level of emergency care is almost entirely missing. In these 15 low-capacity states, the "Safety Net" is incredibly thin: only three states (New Mexico, Maine, and Vermont) have Level 1 hospitals equipped with burn beds, while the rest must rely on lower-level facilities or non-trauma centers like those in Arkansas and Hawaii. This means that in the majority of these states, if a patient has a complex, life-threatening injury, there is no top-tier facility in the entire state capable of treating them. Patients are forced into a dangerous waiting game, relying on smaller hospitals to stabilize them before they can be flown across state lines to find a Level 1 center that actually has an open bed.

In [18]:
# 1. Define the specific states you are interested in
target_states = ['New Mexico', 'Maine', 'Vermont', 'Connecticut', 'West Virginia', 'Arkansas', 'Hawaii']

# 2. Filter for these states AND only hospitals with burn beds
hospitals_with_beds = data[
    (data['STATE_FULL'].isin(target_states)) &
    (data['BURN_BEDS'] > 0)
].copy()

# 3. Apply the label to see their designation
def get_trauma_label(row):
    if row['ADULT_TRAUMA_L1'] > 0: return 'Level 1 Trauma'
    if row['ADULT_TRAUMA_L2'] > 0: return 'Level 2 Trauma'
    return 'Non-Trauma / Other'

hospitals_with_beds['Designation'] = hospitals_with_beds.apply(get_trauma_label, axis=1)

# 4. Select and clean up the columns for the final list
final_list = hospitals_with_beds[[
    'STATE_FULL',
    'Designation',
    'HOSPITAL_NAME',
    'BURN_BEDS'
]].sort_values(by=['Designation', 'STATE_FULL'])

print("Hospitals with Burn Beds in Target States:")
print(final_list.to_string(index=False))

Hospitals with Burn Beds in Target States:
   STATE_FULL        Designation                                                                HOSPITAL_NAME  BURN_BEDS
        Maine     Level 1 Trauma                        Maine Medical Center (Maine Medical Center Burn Unit)        6.0
   New Mexico     Level 1 Trauma                 University of New Mexico Hospital (UNM Regional Burn Center)       10.0
      Vermont     Level 1 Trauma The University of Vermont Medical Center (University of Vermont Burn Center)        9.0
  Connecticut     Level 2 Trauma                            Bridgeport Hospital (The Connecticut Burn Center)        9.0
West Virginia     Level 2 Trauma      Cabell Huntington Hospital (Cabell Huntington Burn Intensive Care Unit)        4.0
     Arkansas Non-Trauma / Other      Arkansas Children's Hospital (Arkansas Children's Hospital Burn Center)       10.0
       Hawaii Non-Trauma / Other                                                           Straub Burn Center 

 If someone is severely burned, even the "best" hospitals in the state (the L1s) are not equipped to keep them. They must be flown out of state. This is exactly the "Burn Desert" problem mentioned in the research papers.

# Burn beds VS Burn_Adult VS Burn_Peds

In [19]:
# Create a check column to see if beds are Adult, Peds, or Both
def check_bed_type(row):
    if row['BURN_ADULT'] > 0 and row['BURN_PEDS'] > 0:
        return "Mixed (Adult & Peds)"
    elif row['BURN_ADULT'] > 0:
        return "Adult Only"
    elif row['BURN_PEDS'] > 0:
        return "Pediatric Only"
    else:
        return "No Burn Beds"

data['Bed_Specialty'] = data.apply(check_bed_type, axis=1)

# Look at the results for your target states
display(data[data['BURN_BEDS'] > 0][['STATE', 'HOSPITAL_NAME', 'BURN_BEDS', 'BURN_ADULT', 'BURN_PEDS', 'Bed_Specialty']])

,STATE,HOSPITAL_NAME,BURN_BEDS,BURN_ADULT,BURN_PEDS,Bed_Specialty
3,AL,University of Alabama at Birmingham Hospital (...,28.0,1.0,0.0,Adult Only
4,AL,Children's of Alabama (Children's of Alabama B...,6.0,0.0,1.0,Pediatric Only
6,AL,USA Health University Hospital (Arnold Luterma...,18.0,1.0,0.0,Adult Only
18,AZ,Valleywise Health (Arizona Burn Center),45.0,1.0,1.0,Mixed (Adult & Peds)
20,AZ,Banner University Medical Center - Tucson (Bur...,35.0,1.0,1.0,Mixed (Adult & Peds)
...,...,...,...,...,...,...
609,WA,UW Medicine/Harborview Medical Center (UW Medi...,40.0,1.0,1.0,Mixed (Adult & Peds)
616,WV,Cabell Huntington Hospital (Cabell Huntington ...,4.0,1.0,0.0,Adult Only
624,WI,University of Wisconsin Hospitals (UW Burn and...,11.0,1.0,1.0,Mixed (Adult & Peds)
628,WI,Children's Wisconsin (Children's Wisconsin Bur...,5.0,0.0,1.0,Pediatric Only


In [20]:
# Filter for Texas ('TX') and ensure they actually have beds
texas_burn_centers = data[(data['STATE'] == 'ND') & (data['BURN_BEDS'] > 0)]

# Display the specific columns you want
display(texas_burn_centers[['STATE', 'HOSPITAL_NAME', 'BURN_BEDS', 'BURN_ADULT', 'BURN_PEDS', 'Bed_Specialty']])

,STATE,HOSPITAL_NAME,BURN_BEDS,BURN_ADULT,BURN_PEDS,Bed_Specialty


# Burn beds by Region for Adult vs peds

In [21]:
# 1. Define the Regions
region_map = {
    'Connecticut': 'Northeast', 'Maine': 'Northeast', 'Massachusetts': 'Northeast', 'New Hampshire': 'Northeast',
    'Rhode Island': 'Northeast', 'Vermont': 'Northeast', 'New Jersey': 'Northeast', 'New York': 'Northeast',
    'Pennsylvania': 'Northeast',
    'Illinois': 'Midwest', 'Indiana': 'Midwest', 'Iowa': 'Midwest', 'Kansas': 'Midwest', 'Michigan': 'Midwest',
    'Minnesota': 'Midwest', 'Missouri': 'Midwest', 'Nebraska': 'Midwest', 'North Dakota': 'Midwest',
    'Ohio': 'Midwest', 'South Dakota': 'Midwest', 'Wisconsin': 'Midwest',
    'Alabama': 'South', 'Arkansas': 'South', 'Delaware': 'South', 'District of Columbia': 'South',
    'Florida': 'South', 'Georgia': 'South', 'Kentucky': 'South', 'Louisiana': 'South', 'Maryland': 'South',
    'Mississippi': 'South', 'North Carolina': 'South', 'Oklahoma': 'South', 'South Carolina': 'South',
    'Tennessee': 'South', 'Texas': 'South', 'Virginia': 'South', 'West Virginia': 'South',
    'Alaska': 'West', 'Arizona': 'West', 'California': 'West', 'Colorado': 'West', 'Hawaii': 'West',
    'Idaho': 'West', 'Montana': 'West', 'Nevada': 'West', 'New Mexico': 'West', 'Oregon': 'West',
    'Utah': 'West', 'Washington': 'West', 'Wyoming': 'West'
}

# 2. Apply cleaning and mapping
data['STATE_FULL'] = data['STATE_FULL'].str.strip()
data['Region'] = data['STATE_FULL'].map(region_map)

# 3. Aggregate data by Region
region_capacity = data.groupby('Region').agg({
    'BURN_BEDS': 'sum',
    'BURN_ADULT': 'sum',
    'BURN_PEDS': 'sum',
    'HOSPITAL_NAME': 'count'
}).reset_index()

# Rename for clarity
region_capacity.rename(columns={'HOSPITAL_NAME': 'Total_Hospitals'}, inplace=True)

# 4. Sort by Total Burn Beds
region_capacity = region_capacity.sort_values(by='BURN_BEDS', ascending=False)

print("Burn Bed Capacity by US Region:")
print(region_capacity.to_string(index=False))

# --- Visualization: Regional Comparison ---
import plotly.express as px

fig = px.bar(
    region_capacity,
    x='Region',
    y=['BURN_ADULT', 'BURN_PEDS'],
    title="Burn Bed Distribution: Adult vs. Pediatric by Region",
    labels={'value': 'Total Beds', 'variable': 'Age Group'},
    color_discrete_map={'BURN_ADULT': '#a50f15', 'BURN_PEDS': '#fc9272'},
    barmode='group'
)
fig.show()

Burn Bed Capacity by US Region:
   Region  BURN_BEDS  BURN_ADULT  BURN_PEDS  Total_Hospitals
    South      847.0        39.0       24.0              188
  Midwest      513.0        33.0       29.0              198
     West      406.0        25.0       23.0              135
Northeast      314.0        23.0       13.0              114


# Pediatric

In [22]:
# 1. Sum Pediatric Burn Beds by State
peds_state_total = data.groupby('STATE_FULL')['BURN_PEDS'].sum().reset_index()

# 2. Sort by Burn Beds (Ascending to show the gaps first)
peds_state_total = peds_state_total.sort_values(by='BURN_PEDS', ascending=True)

# 3. Rename columns for clarity
peds_state_total.columns = ['State Name', 'Total Pediatric Burn Center']

print("Total Pediatric Burn Beds by State:")
print(peds_state_total.to_string(index=False))

# --- Quick Insight for your Presentation ---
zero_peds_states = peds_state_total[peds_state_total['Total Pediatric Burn Center'] == 0]
print(f"\nCRITICAL GAP: There are {len(zero_peds_states)} states with ZERO specialized pediatric burn beds.")


Total Pediatric Burn Beds by State:
          State Name  Total Pediatric Burn Center
              Alaska                          0.0
            Delaware                          0.0
         Connecticut                          0.0
       New Hampshire                          0.0
         Mississippi                          0.0
             Montana                          0.0
               Maine                          0.0
            Kentucky                          0.0
       West Virginia                          0.0
             Vermont                          0.0
        South Dakota                          0.0
        North Dakota                          0.0
               Idaho                          1.0
           Louisiana                          1.0
       Massachusetts                          1.0
             Alabama                          1.0
          New Mexico                          1.0
          New Jersey                          1.0
            Ne

In [23]:
# 2. FILTER: Only look at hospitals that have PEDIATRIC burn beds
# This excludes Adult-only hospitals from the count
peds_facility_data = data[data['BURN_PEDS'] > 0].copy()

# 3. GROUP BY STATE: Sum the TOTAL beds in those Pediatric-capable facilities
state_peds_total_beds = peds_facility_data.groupby('STATE_FULL')['BURN_BEDS'].sum().reset_index()

# 4. MAP: 2-letter codes
state_peds_total_beds['state_code'] = state_peds_total_beds['STATE_FULL'].map(us_state_to_abbrev)

# 5. PLOT: Total Burn Beds associated with Pediatric Burn care
fig_peds_total = px.choropleth(
    state_peds_total_beds,
    locations='state_code',
    locationmode='USA-states',
    color='BURN_BEDS',
    scope='usa',
    # Using Orange-Red (OrRd) to keep the "Pediatric" theme distinct
    color_continuous_scale="OrRd",
    range_color=[0, state_peds_total_beds['BURN_BEDS'].max()],
    labels={'BURN_BEDS': 'Total Beds in Peds Facilities'},
    title='Total Burn Bed Capacity in Facilities with Pediatric Burn Units'
)

fig_peds_total.update_layout(
    geo=dict(lakecolor='rgb(255, 255, 255)'),
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig_peds_total.show()

# Burn Adult

In [24]:
# 1. Sum Adult Burn Beds by State
adult_state_total = data.groupby('STATE_FULL')['BURN_ADULT'].sum().reset_index()

# 2. Sort by Burn Beds (Ascending to show the gaps first)
adult_state_total = adult_state_total.sort_values(by='BURN_ADULT', ascending=True)

# 3. Rename columns for clarity
adult_state_total.columns = ['State Name', 'Total Adult Burn Beds']

print("Total Adult Burn Beds by State:")
print(adult_state_total.to_string(index=False))

# --- Quick Insight for your Presentation ---
zero_adult_states = adult_state_total[adult_state_total['Total Adult Burn Beds'] == 0]
print(f"\nCRITICAL GAP: There are {len(zero_adult_states)} states with ZERO specialized adult burn beds.")

Total Adult Burn Beds by State:
          State Name  Total Adult Burn Beds
              Alaska                    0.0
            Delaware                    0.0
       New Hampshire                    0.0
             Montana                    0.0
         Mississippi                    0.0
        North Dakota                    0.0
        South Dakota                    0.0
               Idaho                    1.0
          New Mexico                    1.0
            Maryland                    1.0
               Maine                    1.0
            Nebraska                    1.0
            Kentucky                    1.0
                Iowa                    1.0
         Connecticut                    1.0
            Arkansas                    1.0
          New Jersey                    1.0
      South Carolina                    1.0
             Vermont                    1.0
                Utah                    1.0
              Oregon                    1.0


In [25]:
# 2. FILTER: Only look at hospitals that have ADULT burn beds
# This excludes Pediatric-only hospitals from the count
adult_facility_data = data[data['BURN_ADULT'] > 0].copy()

# 3. GROUP BY STATE: Sum the TOTAL beds in those Adult facilities
state_adult_total_beds = adult_facility_data.groupby('STATE_FULL')['BURN_BEDS'].sum().reset_index()

# 4. MAP: 2-letter codes (Ensure us_state_to_abbrev is defined)
state_adult_total_beds['state_code'] = state_adult_total_beds['STATE_FULL'].map(us_state_to_abbrev)

# 5. PLOT: Total Burn Beds associated with Adult Burn care
fig_adult_total = px.choropleth(
    state_adult_total_beds,
    locations='state_code',
    locationmode='USA-states',
    color='BURN_BEDS',
    scope='usa',
    color_continuous_scale="Reds",
    range_color=[0, state_adult_total_beds['BURN_BEDS'].max()],
    labels={'BURN_BEDS': 'Total Beds in Adult Units'},
    title='Total Burn Bed Capacity in Facilities with Adult Burn Units'
)

fig_adult_total.update_layout(
    geo=dict(lakecolor='rgb(255, 255, 255)'),
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig_adult_total.show()

# Conclusion

Basically, our burn care system is a mess because it's lopsided. Nearly one-third of the country has almost no specialized beds. It’s even worse for kids—they are 70% more likely to live in a state with zero help. This forces families to fly across the country for life-saving care, which is expensive, scary, and puts too much pressure on the few 'Hub' hospitals that are left to do all the work